In [ ]:
import sys, os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
import torch
import torch.nn as nn
import torch.nn.functional as F
from src import *

# Setup

In [ ]:
# load MNIST data
train_img_path = "../data/train_img.idx"
train_label_path = "../data/train_label.idx"
test_img_path = "../data/test_img.idx"
test_label_path = "../data/test_label.idx"

train_samples, train_labels, test_samples, test_labels = normalize_mnist_data(
    train_img_path, train_label_path, test_img_path, test_label_path
)

# use tensors
train_samples = torch.from_numpy(train_samples).float()
train_labels = torch.from_numpy(train_labels)
test_samples = torch.from_numpy(test_samples).float()
test_labels = torch.from_numpy(test_labels)

In [ ]:
# hyperparameters
epochs = 100
batch_size = 600
lr = 3e-4
channels = 12
npix = 784
nout = 10

# initialize model

m = nn.Sequential(
    nn.Conv2d(1, channels, kernel_size=3, stride=1, padding=2),
    nn.ReLU(),
    nn.MaxPool2d(2, 1),
    nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, 1),
    nn.Flatten(),
    nn.Linear(channels * npix, nout),  # Adjusted input features for Linear layer
    nn.Softmax(-1),
)

optimizer = torch.optim.AdamW(m.parameters(), lr=lr)

# Training Loop

In [ ]:
# train model

for epoch in range(epochs):
    # randomize order of training samples and labels
    idx = torch.randperm(len(train_samples))
    train_samples = train_samples[idx]
    train_labels = train_labels[idx]
    epoch_loss = 0.0
    for i in range(0, len(train_samples), batch_size):
        optimizer.zero_grad()
        batch_samples = train_samples[i : i + batch_size].view(-1, 1, 28, 28)
        batch_labels = train_labels[i : i + batch_size].flatten()
        y_hat = m(batch_samples)
        loss = F.cross_entropy(y_hat, batch_labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch + 1} loss: {epoch_loss}")
    epoch_loss = 0.0

In [ ]:
# save model parameters
torch.save(m.state_dict(), "../data/cnn_mnist_model.pth")

# Train Accuracy

In [ ]:
correct = 0
for i in range(len(train_samples)):
    y_hat = m(train_samples[i].view(-1, 1, 28, 28))
    correct += (torch.argmax(y_hat) == train_labels[i]).item()

print(f"accuracy: {correct / len(train_samples) * 100}%")

# Validation Accuracy

In [ ]:
# test model accuracy
correct = 0
for i in range(len(test_samples)):
    y_hat = m(test_samples[i].view(-1, 1, 28, 28))
    correct += (torch.argmax(y_hat) == test_labels[i]).item()

print(f"accuracy: {correct / len(test_samples) * 100}%")